In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

os.listdir(path) # to check the file name


In [ ]:
# Task 1:Read the dataset `Q1_data.csv` using `read_csv()`

import pandas as pd
# Load the dataset
csv_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Inspect the first few rows using `head()`

df.head() # print first 5 row

In [ ]:
# Task 3: Display dataset information using `info()`

df.info() # Check data types and structure

In [ ]:
# Task 4: Show statistical description using `describe()`

df.describe() # Descriptive statistics for numerical columns


In [ ]:
# Task 5: Plot the target distribution (delivery_time)

import matplotlib.pyplot as plt

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Drop the 'Order_ID' column from the data
df = df.drop(columns=['Order_ID'])
df.head()


In [ ]:
# Task 2: Handle missing values appropriately(Hint: I guess you want to have a closer look at the columns with missing values :) )

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
df= df.dropna()
check_missing_values(df)


In [ ]:
# Task 3: 3. Check and remove duplicates if any exist

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 4: 4. Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le


df.head()


In [ ]:
# Task 5: Apply feature scaling for all features** (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Check for target imbalance and state if it is imbalanced or not** (keep this cell empty if not needed)

print("Target Distribution:")
print(df['Delivery_Time'].value_counts(normalize=True))

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error
import numpy as np

model = RandomForestRegressor(n_estimators=200)

# 2.Use the correct split: **KFold** OR **StratifiedKFold**
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_mae = []

# Storage for linear regression results for each fold

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]

  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Train a **RandomForest** model
  model.fit(X_train, y_train)

# Use the model to predict the test data
  y_pred = model.predict(X_test)

#4. Evaluate using **MAE (Mean Absolute Error) ONLY**
  mae = mean_absolute_error(y_test, y_pred)
  lr_mae.append(mae)

# 5. Print the averaged score across all folds
print(" Random Forest Results")
print(f"  Average MAE: {np.mean(lr_mae):.4f}")


In [ ]:
# Task 1: 1. Plot feature importance from your trained model
import pandas as pd
import matplotlib.pyplot as plt
feature_cols = X.columns
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Plot predicted delivery time histogram

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black', color='green')
plt.title('predicted delivery time')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task Bonus: Write your code here: